In [11]:
import os
import json
import torch
import numpy as np
from dataset import EurDataset, collate_data
from models.transceiver import DeepSC
from torch.utils.data import DataLoader
from utils import BleuScore, SNR_to_noise, greedy_decode, SeqtoText
from tqdm import tqdm
from sklearn.preprocessing import normalize
from w3lib.html import remove_tags

from transformers import AutoTokenizer, AutoModelForSequenceClassification

DATA_DIR = 'txt/train_data.pkl'
VOCAB_FILE = 'data/txt/vocab.json'
CHECKPOINT_PATH = 'checkpoints/deepsc-Rayleigh'
CHANNEL = 'Rayleigh'
MAX_LENGTH = 30
MIN_LENGTH = 4
D_MODEL = 128
DFF = 512
NUM_LAYERS = 4
NUM_HEADS = 8
BATCH_SIZE = 64
EPOCHS = 2
BERT_CONFIG_PATH = 'bert/cased_L-12_H-768_A-12/bert_config.json'
BERT_CHECKPOINT_PATH = 'bert/cased_L-12_H-768_A-12/bert_model.ckpt'
BERT_DICT_PATH = 'bert/cased_L-12_H-768_A-12/vocab.txt'

SNR = [0, 3, 6, 9, 12, 15, 18]

device = torch.device("cpu")

# Load tokenizer and model fine-tuned on MNLI
model_name = "roberta-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3051.72it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, b

In [ ]:
def performance(snr_list, net):
    bleu_score_1gram = BleuScore(1, 0, 0, 0)

    test_eur = EurDataset('test')
    test_iterator = DataLoader(test_eur, batch_size=BATCH_SIZE, num_workers=0,
                               pin_memory=True, collate_fn=collate_data)

    StoT = SeqtoText(token_to_idx, end_idx)
    score = []
    nli_score = []
    score2 = []
    net.eval()
    with torch.no_grad():
        for epoch in range(EPOCHS):
            received = []
            transmitted = []

            for snr in tqdm(snr_list):
                word = []
                target_word = []
                noise_std = SNR_to_noise(snr)

                # iterator returns a batch with mutiple sentences
                for batch_idx, batch in enumerate(test_iterator):
                    if batch_idx % 100000 != 0:
                        continue

                    batch = batch.to(device)
                    target = batch

                    out = greedy_decode(net, batch, noise_std, MAX_LENGTH, pad_idx,
                                        start_idx, CHANNEL)

                    sentences = out.cpu().numpy().tolist()
                    result_string = list(map(StoT.sequence_to_text, sentences))
                    word = word + result_string
                    print(np.array(sentences).shape)

In [15]:
vocab = json.load(open(VOCAB_FILE, 'rb'))
token_to_idx = vocab['token_to_idx']
idx_to_token = dict(zip(token_to_idx.values(), token_to_idx.keys()))
num_vocab = len(token_to_idx)
pad_idx = token_to_idx["<PAD>"]
start_idx = token_to_idx["<START>"]
end_idx = token_to_idx["<END>"]

""" define optimizer and loss function """
deepsc = DeepSC(NUM_LAYERS, num_vocab, num_vocab,
                    num_vocab, num_vocab, D_MODEL, NUM_HEADS,
                    DFF, 0.1).to(device)

checkpoint = torch.load("checkpoints/deepsc-Rayleigh/checkpoint_80_europarl.pth", map_location=device)
deepsc.load_state_dict(checkpoint)
print('model load!')

bleu_score = performance(SNR, deepsc)


model load!


  0%|          | 0/7 [00:00<?, ?it/s]/home/ishaang/python/dd/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
 14%|█▍        | 1/7 [00:06<00:41,  6.98s/it]

64


 14%|█▍        | 1/7 [00:14<01:27, 14.51s/it]


KeyboardInterrupt: 